In [1]:
"""
Step-count analysis: do I walk more on working days than days off?
===================================================================

Data sources
------------
1. Garmin daily step totals, taken from the full Garmin account export
   (GDPR "Export Your Data" request -> DI_CONNECT/DI-Connect-Aggregator/
   UDSFile_*.json). Chosen over Apple Health because the watch is worn all
   day and its history is complete at source, whereas the Health sync was
   patchy (only 34 days landed there).
2. Work rota spreadsheets exported to CSV, one file per period, naming
   convention rota_YYYY-MM.csv. These tell me which days were full shifts,
   half shifts, or days off.

Pipeline: parse Garmin JSON -> daily steps -> parse rotas -> map shift
codes to 3 categories -> join steps to rota -> describe + test.

Author: Tim
"""

import json
import zipfile
import re
from pathlib import Path

import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt


# ======================================================================
# PART 1 - GARMIN STEPS
# ======================================================================

def load_garmin_steps(zip_path):
    """Read all daily-summary JSON files from a Garmin export archive and
    return a clean DataFrame of one row per day: date + totalSteps.

    The archive holds ~100-day chunks named UDSFile_START_END.json. Each
    file is a list of daily records with many fields; we only need the
    date and the step count.
    """
    gz = zipfile.ZipFile(zip_path)
    uds_files = [n for n in gz.namelist() if 'UDSFile' in n]

    frames = []
    for name in uds_files:
        with gz.open(name) as f:
            data = json.load(f)
        # json_normalize flattens the nested JSON into a flat table;
        # nested fields become dotted column names.
        frames.append(pd.json_normalize(data))

    garmin = pd.concat(frames, ignore_index=True)

    # Keep only the two columns we care about.
    g = garmin[['calendarDate', 'totalSteps']].copy()
    g['calendarDate'] = pd.to_datetime(g['calendarDate'])
    g = g.sort_values('calendarDate')

    # Data-quality checks that actually mattered here:
    #  - overlapping chunks could duplicate dates
    g = g.drop_duplicates(subset='calendarDate', keep='first')
    #  - Garmin writes no row at all when the watch is not worn / not
    #    synced, so there are no zero-step rows to worry about, but there
    #    ARE calendar gaps. We do not fill them; we just work with the
    #    days we have. Gaps come from irregular syncing, not from the work
    #    schedule, so they are missing-at-random with respect to our
    #    question.
    return g


def steps_since_start(g, start='2025-08-01'):
    """Restrict to the current job. I started this role on 1 Aug 2025;
    earlier data belongs to a different routine and would contaminate the
    comparison (a subtle form of leakage).
    """
    data = g[g['calendarDate'] >= start].copy()
    data['weekday'] = data['calendarDate'].dt.day_name()
    data = data.dropna(subset=['totalSteps'])   # drop empty-step rows
    return data


# ======================================================================
# PART 2 - WORK ROTA
# ======================================================================

# Matches a date cell like "1-Nov" or "23-Feb" anywhere in a string.
DATE_PAT = re.compile(r'(\d{1,2})-([A-Za-z]{3})')


def parse_rota(path):
    """Parse one rota CSV into rows of (date, shift_raw) for me only.

    The rota files are messy and inconsistent between months:
      - some have the date glued to the weekday with a newline,
      - some keep dates on their own row,
      - some are shifted right by an empty column,
      - one "monthly" file actually contains three months.
    So instead of assuming fixed positions we LOCATE the data by shape:
      - the date row is whichever row contains the most "NN-Mon" cells,
      - my row is the first row containing 'Tim' below the date row.
    Files are also cp1252-encoded (Excel), not UTF-8 - hence encoding.
    """
    year = int(path.stem.split('_')[1].split('-')[0])
    raw = pd.read_csv(path, header=None, encoding='cp1252')

    # Find the row that looks most like a row of dates.
    def count_dates(row):
        return sum(bool(DATE_PAT.search(str(v))) for v in row)
    date_row_idx = raw.apply(count_dates, axis=1).idxmax()
    date_row = raw.iloc[date_row_idx]

    # Find my row (contains 'Tim'), preferring one below the date row.
    tim_mask = raw.apply(lambda r: r.astype(str).str.contains('Tim').any(), axis=1)
    tim_below = [i for i in raw.index[tim_mask] if i > date_row_idx]
    tim_row = raw.iloc[tim_below[0]] if tim_below else raw[tim_mask].iloc[0]

    # Pair each dated column with my shift code in that column.
    records = []
    for col in raw.columns:
        m = DATE_PAT.search(str(date_row[col]))
        if not m:
            continue
        day, mon = m.group(1), m.group(2).title()
        try:
            date = pd.to_datetime(f'{day}-{mon}-{year}', format='%d-%b-%Y')
        except ValueError:
            continue
        records.append({'date': date, 'shift_raw': str(tim_row[col]).strip()})
    return pd.DataFrame(records)


# Map every raw shift code to one of three categories.
#   full = full working day
#   half = short shift (about 4 hours from 13:00)
#   off  = non-working day
# Everything else (annual leave, single-digit holiday markers, blanks)
# maps to None and is excluded from the analysis on purpose:
#   - 'AL' / 'A/L'   annual leave (different routine, not a normal day off)
#   - '1'..'5'       holiday-day counters (were red-shaded in the sheet)
#   - 'nan'          unfilled cells
SHIFT_MAP = {
    '9-5': 'full', '9am-5pm': 'full', '10-4': 'full', '9-3pm': 'full',
    '4h -AP': 'half', '4h-AP': 'half', '4h AP': 'half',
    '13-5pm': 'half', '9am-1pm': 'half',
    'NWD': 'off',
    'A/L': None, 'AL': None, 'BTRIP': None,
}


def build_rota(folder='.'):
    """Parse every rota_*.csv in the folder, concatenate, deduplicate,
    and map shift codes to categories."""
    files = sorted(Path(folder).glob('rota_*.csv'))
    rota = pd.concat([parse_rota(f) for f in files], ignore_index=True)

    # Overlapping files repeat some dates. Where the same date carries
    # different codes (e.g. a plan vs a later final version), keep the
    # last file, which is closer to the date and therefore more final.
    rota = rota.drop_duplicates(subset='date', keep='last').sort_values('date')

    # Unknown codes -> None (get(code, None)), so nothing unexpected
    # sneaks in silently mislabelled.
    rota['category'] = rota['shift_raw'].map(lambda c: SHIFT_MAP.get(c, None))
    return rota


# ======================================================================
# PART 3 - JOIN AND ANALYSE
# ======================================================================

def analyse(steps, rota):
    """Inner-join daily steps to rota categories and compare groups."""
    df = steps.rename(columns={'calendarDate': 'date'})[['date', 'totalSteps']]
    df = df.merge(rota[['date', 'category']], on='date', how='inner')
    df = df.dropna(subset=['category'])   # drop leave / blanks / unknown

    print("Labelled days with steps:", len(df))
    print("\nDays per category:")
    print(df['category'].value_counts().to_string())

    print("\nSteps per category:")
    summary = (df.groupby('category')['totalSteps']
                 .agg(['count', 'median', 'mean', 'std']).round(0))
    print(summary.to_string())

    # Main test: full working day vs day off.
    full = df[df['category'] == 'full']['totalSteps']
    off = df[df['category'] == 'off']['totalSteps']
    t, p = stats.ttest_ind(full, off, equal_var=False)   # Welch, unequal var
    u, pu = stats.mannwhitneyu(full, off)                 # non-parametric backup
    print("\nFull working day vs day off:")
    print(f"  medians: {full.median():.0f} vs {off.median():.0f} steps")
    print(f"  difference: {full.median() - off.median():.0f} steps/day")
    print(f"  Welch t-test:   t={t:.2f}, p={p:.4g}")
    print(f"  Mann-Whitney U: U={u:.0f}, p={pu:.4g}")

    return df


def plot_categories(df, path='steps_by_category.png'):
    """Box plot of steps across the three day types, ordered off < half < full."""
    order = ['off', 'half', 'full']
    df = df.copy()
    df['category'] = pd.Categorical(df['category'], categories=order, ordered=True)
    df.boxplot(column='totalSteps', by='category', figsize=(9, 5))
    plt.suptitle('')
    plt.title('Daily steps by day type')
    plt.ylabel('Steps')
    plt.xlabel('Day type')
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.show()


# ======================================================================
# RUN
# ======================================================================

if __name__ == '__main__':
    # 1. Steps. Point this at your Garmin export zip.
    #    (If you already saved steps_daily.csv, load that instead:
    #     steps = pd.read_csv('steps_daily.csv');
    #     steps['calendarDate'] = pd.to_datetime(steps['calendarDate'])
    #     steps = steps.dropna(subset=['totalSteps']) )
    g = load_garmin_steps('Data_garmin.zip')
    steps = steps_since_start(g, start='2025-08-01')

    # 2. Rota. All rota_YYYY-MM.csv files in the current folder.
    rota = build_rota('.')
    print("Raw shift codes after assembly:")
    print(rota['shift_raw'].value_counts().to_string())
    print("\nAfter mapping to categories:")
    print(rota['category'].value_counts(dropna=False).to_string())
    print("\n" + "=" * 50)

    # 3. Join, test, plot.
    df = analyse(steps, rota)
    plot_categories(df)

FileNotFoundError: [Errno 2] No such file or directory: 'Data_garmin.zip'

In [ ]:
plt.savefig('steps_by_category.png', dpi=120, bbox_inches='tight')

<Figure size 640x480 with 0 Axes>